# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is not set."

print("HF token loaded successfully.")

In [ ]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March 2026 rows:", len(march_df))

### Observation window and grain verification

In [ ]:
# Verify date span
print("Minimum report date:", march_df["report_date"].min())
print("Maximum report date:", march_df["report_date"].max())
print("Unique report dates:", march_df["report_date"].nunique())

# Verify grain: each row should be unique on (report_date, client_hash_id, content_hash_id)
grain_counts = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
)
print("Unique date/client/content combinations:", len(grain_counts))
print("Max rows per combination:", grain_counts["row_count"].max())
print("Duplicate combinations:", (grain_counts["row_count"] > 1).sum())

### GSC and GA4 data availability

In [ ]:
import duckdb

availability = duckdb.query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS gsc_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS ga4_pct
    FROM march_df
""").df()

print("GSC/GA4 availability in March 2026 partition:")
print(f"  GSC available: {int(availability['gsc_available'].iloc[0]):,} / {int(availability['total_rows'].iloc[0]):,} rows ({availability['gsc_pct'].iloc[0]}%)")
print(f"  GA4 available: {int(availability['ga4_available'].iloc[0]):,} / {int(availability['total_rows'].iloc[0]):,} rows ({availability['ga4_pct'].iloc[0]}%)")

### Build the five-feature frame

Aggregate March 2026 daily rows to page/client grain. CTR is computed from the aggregated totals (clicks / impressions); when impressions are zero, CTR is set to NA.

In [ ]:
page_features = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_gsc_impressions=("gsc_impressions", "sum"),
        march_gsc_clicks=("gsc_clicks", "sum"),
        march_ga4_sessions=("ga4_sessions", "sum"),
        march_scroll_events=("scroll_events", "sum"),
    )
)

# CTR: clicks / impressions, NA when impressions == 0
page_features["march_gsc_ctr"] = (
    page_features["march_gsc_clicks"]
    / page_features["march_gsc_impressions"].replace(0, pd.NA)
)

feature_cols = [
    "march_gsc_impressions",
    "march_gsc_clicks",
    "march_gsc_ctr",
    "march_ga4_sessions",
    "march_scroll_events",
]

feature_frame = page_features[
    ["client_hash_id", "content_hash_id"] + feature_cols
].copy()

print("Feature frame shape:", feature_frame.shape)
print("Unique clients:", feature_frame["client_hash_id"].nunique())
print("Unique content items:", feature_frame["content_hash_id"].nunique())

### Content hash uniqueness check

Verify whether `content_hash_id` is globally unique across all clients, or whether the same content appears under multiple clients (page/client observations).

In [ ]:
# Is content_hash_id globally unique across all clients?
n_rows = len(feature_frame)
n_unique_content = feature_frame["content_hash_id"].nunique()
n_unique_client_content = feature_frame.groupby(["client_hash_id", "content_hash_id"]).ngroups

print(f"Total rows in feature frame: {n_rows:,}")
print(f"Unique content_hash_id (global): {n_unique_content:,}")
print(f"Unique (client_hash_id, content_hash_id) pairs: {n_unique_client_content:,}")
print(f"content_hash_id globally unique: {n_unique_content == n_rows}")

if n_unique_content == n_rows:
    print("\nTerminology: 'unique content items' is correct \u2014 each content_hash_id appears exactly once.")
else:
    content_per_client = feature_frame.groupby("content_hash_id")["client_hash_id"].nunique()
    multi_client = (content_per_client > 1).sum()
    print(f"\ncontent_hash_id appears under multiple clients for {multi_client:,} items.")
    print("Terminology change: use 'page/client observations' instead of 'unique content items'.")


In [ ]:
feature_frame.head(10)

### Missing-value profile

In [ ]:
missing = feature_frame[feature_cols].isna().sum().to_frame("na_count")
missing["pct_missing"] = (missing["na_count"] / len(feature_frame) * 100).round(1)
missing["total_rows"] = len(feature_frame)

print("Missing-value profile:")
print(missing.to_string())

# Note: march_gsc_ctr is NA when impressions == 0 (division by zero guard).
# march_ga4_sessions and march_scroll_events come from GA4; if a page has no GA4
# data available, the warehouse zero-fills these columns. We do NOT blanket-fill NA
# because the underlying warehouse may have set ga4_data_available=FALSE.

### Define the opportunity proxy

In [ ]:
feature_frame["opportunity_proxy"] = (
    (feature_frame["march_gsc_impressions"] > 0)
    & (feature_frame["march_gsc_clicks"] == 0)
).astype(int)

print("Opportunity proxy distribution:")
print(feature_frame["opportunity_proxy"].value_counts())
print(f"\nBase rate (opportunity_proxy == 1): {feature_frame['opportunity_proxy'].mean():.1%}")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
feature_notes = pd.DataFrame({
    "feature": [
        "march_gsc_impressions",
        "march_gsc_clicks",
        "march_gsc_ctr",
        "march_ga4_sessions",
        "march_scroll_events",
    ],
    "meaning": [
        "Total GSC search impressions for this page across March 2026",
        "Total GSC search clicks for this page across March 2026",
        "Click-through rate: march_gsc_clicks / march_gsc_impressions",
        "Total GA4 sessions for this page across March 2026",
        "Total GA4 scroll events for this page across March 2026",
    ],
    "type": ["numeric"] * 5,
    "missingness": [
        "None \u2014 zero when page had no GSC impressions",
        "None \u2014 zero when page had no GSC clicks",
        "NA when impressions == 0 (div-by-zero guard)",
        "Zero-filled by warehouse when GA4 unavailable (check ga4_data_available flag)",
        "Zero-filled by warehouse when GA4 unavailable (check ga4_data_available flag)",
    ],
    "available_at_decision_time": [
        "Yes \u2014 March GSC data is observed before the prioritisation decision",
        "Yes \u2014 March GSC data is observed before the prioritisation decision",
        "Yes \u2014 derived from March impressions and clicks, both observed",
        "Yes when ga4_data_available is TRUE \u2014 otherwise unreliable (zero-filled placeholder)",
        "Yes when ga4_data_available is TRUE \u2014 otherwise unreliable (zero-filled placeholder)",
    ],
    "caveat": [
        "Pages with zero impressions may never have appeared in search results",
        "Pages with zero clicks may still have had search impressions (the proxy's basis)",
        "Not defined when impressions == 0; use with caution for pages with low impression counts",
        "GA4 data is not available for all clients \u2014 zero may mean 'no GA4' not 'no sessions'",
        "GA4 scroll events unavailable for same reason as sessions; not directly comparable to GSC data",
    ],
})

print("Feature notes table:")
print(feature_notes.to_string(index=False))

In [ ]:
# GA4 availability detail: how many page-level rows are affected?

# The daily fact has ga4_data_available per row; after aggregation to page-level,
# a page may have SOME March days with GA4 and some without.
# Count daily rows with ga4_data_available = FALSE in March
ga4_unavailable_days = march_df["ga4_data_available"].value_counts()
print("Daily rows with GA4 available (March 2026):")
print(ga4_unavailable_days)
ga4_true_count = march_df["ga4_data_available"].sum()
ga4_false_count = march_df["ga4_data_available"].eq(False).sum()
print(f"\nGA4 unavailable rate: {ga4_false_count / len(march_df):.1%}")

### Client data-start alignment

Check how many clients have `gsc_data_start` or `ga4_data_start` later than 2026-03-01, and how incomplete-history clients relate to the 331,437-row feature frame.

In [ ]:
dim_clients_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)
dim_clients = pd.read_parquet(dim_clients_file)

print(f"dim_clients rows: {len(dim_clients)}")
print(f"Columns: {list(dim_clients.columns)}")
print()
print(dim_clients[["client_hash_id", "gsc_data_start", "ga4_data_start"]].to_string(index=False))

In [ ]:
# How many clients have data_start later than 2026-03-01?
cutoff = pd.Timestamp("2026-03-01")

gsc_late = dim_clients[pd.to_datetime(dim_clients["gsc_data_start"]) > cutoff]
ga4_late = dim_clients[pd.to_datetime(dim_clients["ga4_data_start"]) > cutoff]

print(f"Clients with gsc_data_start after 2026-03-01: {len(gsc_late)} / {len(dim_clients)}")
if len(gsc_late) > 0:
    print(gsc_late[["client_hash_id", "gsc_data_start"]].to_string(index=False))

print(f"\nClients with ga4_data_start after 2026-03-01: {len(ga4_late)} / {len(dim_clients)}")
if len(ga4_late) > 0:
    print(ga4_late[["client_hash_id", "ga4_data_start"]].to_string(index=False))

feature_clients = set(feature_frame["client_hash_id"].unique())
late_gsc_clients = set(gsc_late["client_hash_id"])
late_ga4_clients = set(ga4_late["client_hash_id"])

affected_by_gsc = feature_clients & late_gsc_clients
affected_by_ga4 = feature_clients & late_ga4_clients

rows_affected_gsc = feature_frame[feature_frame["client_hash_id"].isin(affected_by_gsc)].shape[0]
rows_affected_ga4 = feature_frame[feature_frame["client_hash_id"].isin(affected_by_ga4)].shape[0]

print(f"\nFeature frame clients affected by late GSC start: {len(affected_by_gsc)} clients, {rows_affected_gsc:,} rows ({rows_affected_gsc/len(feature_frame)*100:.1f}%)")
print(f"Feature frame clients affected by late GA4 start: {len(affected_by_ga4)} clients, {rows_affected_ga4:,} rows ({rows_affected_ga4/len(feature_frame)*100:.1f}%)")

print("\nNote: clients with late data starts have incomplete March history.")
print("Their March aggregates may undercount true monthly performance.")
print("This is a data-availability limitation, not a design flaw \u2014 documented here for transparency.")

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### A. Label-derived leakage

The opportunity proxy is defined as `(march_gsc_impressions > 0) & (march_gsc_clicks == 0)`. Two of the five features \u2014 `march_gsc_impressions` and `march_gsc_clicks` \u2014 are the direct inputs to this definition. Using them as features is not "future leakage" (they are observed at decision time), but a model that learns from them can only rediscover the mechanical rule that produced the proxy. The remaining three features (`march_gsc_ctr`, `march_ga4_sessions`, `march_scroll_events`) are not components of the proxy and can carry independent signal.

I demonstrate the leakage concept by scoring with the proxy itself (a leaked score), then remove it.

> **Note:** The results below are **in-sample diagnostics**. They demonstrate the mechanical relationship between features and proxy, but do NOT represent out-of-sample predictive performance. See Section 5 for genuine held-out evaluation.

In [ ]:
K = 1000

# LEAKED score: directly uses the proxy itself as a scoring feature
feature_frame["leaked_score"] = feature_frame["opportunity_proxy"]

top_k_leaked = feature_frame.nlargest(K, "leaked_score")
leaked_precision = top_k_leaked["opportunity_proxy"].mean()

print(f"Leaked Precision@{K}: {leaked_precision:.3f}")
print(f"Base rate: {feature_frame['opportunity_proxy'].mean():.1%}")
print(f"Lift over base rate: {leaked_precision / feature_frame['opportunity_proxy'].mean():.1f}x")

# Remove the leaked column
feature_frame = feature_frame.drop(columns=["leaked_score"])

The leaked score achieves **Precision@1000 of 1.000** \u2014 a perfect score because the scoring rule is the label itself. This confirms that using the proxy as a feature is pure leakage.

### Train-without / feature-removal experiment

The skill asks for a train-without test. Here I remove `march_gsc_impressions` and `march_gsc_clicks` (the proxy's mechanical inputs) and score using only the remaining three features. This tests whether the other features carry independent signal beyond the mechanical proxy relationship.

> **Note:** These results are **in-sample diagnostics** scored on the full dataset.

In [ ]:
# Honest score using all five features (impressions / clicks + 1)
feature_frame["honest_score_all5"] = (
    feature_frame["march_gsc_impressions"]
    / (feature_frame["march_gsc_clicks"] + 1)
)

top_k_all5 = feature_frame.nlargest(K, "honest_score_all5")
precision_all5 = top_k_all5["opportunity_proxy"].mean()

print(f"Honest Precision@{K} (all 5 features): {precision_all5:.3f}")
print(f"Base rate: {feature_frame['opportunity_proxy'].mean():.1%}")

# Score using ONLY the three features that are NOT proxy inputs
# CTR is NA for zero-impression pages; fill with 0 for this experiment.
ctr_filled = feature_frame["march_gsc_ctr"].fillna(0).astype(float)

no_ic_score = (
    ctr_filled
    + feature_frame["march_ga4_sessions"].astype(float)
    + feature_frame["march_scroll_events"].astype(float)
)

top_k_no_ic = feature_frame.assign(_score=no_ic_score).nlargest(K, "_score")
precision_no_ic = top_k_no_ic["opportunity_proxy"].mean()

print(f"Honest Precision@{K} (without impressions/clicks): {precision_no_ic:.3f}")
print(f"Base rate: {feature_frame['opportunity_proxy'].mean():.1%}")
print(f"\nInterpretation: removing impressions/clicks reduces precision, confirming")
print(f"that those two features carry the strongest mechanical link to the proxy.")
print(f"The non-zero precision without them shows some independent signal exists")
print(f"in CTR, GA4 sessions, and scroll events.")

# Clean up experiment columns
feature_frame = feature_frame.drop(columns=["honest_score_all5"])

**Key finding:** Removing `march_gsc_impressions` and `march_gsc_clicks` does not collapse the score to zero \u2014 the remaining features carry some independent signal. However, the drop in precision confirms that impressions and clicks have the strongest mechanical relationship to the proxy. This is expected: the proxy is literally defined from these columns. In a production model, the decision about whether to include impressions and clicks depends on the business question \u2014 they are legitimate decision-time features, but their predictive power is partially mechanical.

### B. Future / overlapping windows

I verify that no April 2026 information enters the March feature vector by checking the actual date range of the source partition and the construction logic.

In [ ]:
# The March partition was loaded explicitly as month=2026-03.
# Verify that the data contains only March dates.
date_range = march_df["report_date"].agg(["min", "max"])
print(f"Data covers: {date_range['min']} to {date_range['max']}")

# Check that no April rows snuck in
report_dates = pd.to_datetime(march_df["report_date"])
non_march_rows = march_df[~report_dates.between("2026-03-01", "2026-03-31")]
print(f"Non-March rows in March partition: {len(non_march_rows)}")
print(f"All dates in March 2026: {report_dates.min().month == 3 and report_dates.max().month == 3}")

In [ ]:
# The feature frame construction uses only columns from the March partition:
# gsc_impressions, gsc_clicks, ga4_sessions, scroll_events
# All aggregated within the March date range. No future columns are joined.

# Verify no April-related columns exist in the March dataframe
april_cols = [c for c in march_df.columns if "april" in c.lower() or "2026-04" in c]
print(f"April-related columns in March dataframe: {april_cols}")
print("\nConclusion: No April information enters the March feature vector.")
print("The feature vector is built exclusively from March 2026 daily observations.")

### Explicit future-window timeline

| Component | Window | Status |
|---|---|---|
| **Decision moment** | End of March 2026 | We prioritise pages at the end of March |
| **Feature window** | March 1\u201331, 2026 | All five features are aggregated from this window |
| **Proxy/label window** | March 1\u201331, 2026 | `opportunity_proxy` is defined from the same March window |
| **April/May data** | Not loaded, not joined, not used | Verified above \u2014 no April columns exist |

**Critical finding:** The current proxy and features are in the **same** March window. This is **not** future leakage from April \u2014 no future data enters the features. However, it creates a **mechanical/overlapping-window relationship**: the features and the label are computed from identical underlying daily observations. A model scoring `march_gsc_impressions / (march_gsc_clicks + 1)` is not learning to predict \u2014 it is rediscovering the rule that created the proxy. This limits any interpretation of the observed Precision@K as genuine predictive performance.

### C. Decision-derived / product flags

I verify that no product flags, action labels, priority scores, reason codes, or existing-system outputs are used as model features. The five-feature contract contains only raw March performance metrics \u2014 no downstream decision artifacts.

In [ ]:
# Check the warehouse columns that would represent decision-derived fields
decision_like_cols = [
    c for c in march_df.columns
    if any(kw in c.lower() for kw in [
        "priority", "action", "flag", "score", "label", "tier",
        "recommendation", "baseline", "status", "reason"
    ])
]

print("Warehouse columns that could be decision-derived:")
print(decision_like_cols if decision_like_cols else "  (none)")

# Verify none of the five features are decision-derived
print(f"\nFive features in the contract: {feature_cols}")
print("None of these are product flags, action labels, or existing-system scores.")
print("They are raw March performance metrics available at the decision moment.")

In [ ]:
# The query table (fact_content_query_90d) is NOT used by the five-feature contract.
# It provides query-mix features (diversity, concentration, rare/anonymized shares)
# that could be useful but are excluded from this scope.

print("Query table usage: EXCLUDED")
print("The five-feature contract does not use fact_content_query_90d.")
print("Query-mix features (diversity, concentration, tail shares) are out of scope.")

### Leakage summary

| Check | Result |
|---|---|
| Label-derived leakage | `march_gsc_impressions` and `march_gsc_clicks` are proxy inputs \u2014 mechanical dependence, not future leakage. Leaked score (proxy as feature) achieves Precision@1000 = 1.000. Removing them reduces precision but does not collapse to zero. |
| Future / overlapping windows | No April data enters March features. Source partition is explicitly March 2026 only. The proxy and features share the same March window \u2014 this is mechanical overlap, not future leakage. |
| Decision-derived / product flags | No product flags, action labels, priority scores, or reason codes are used as features. The five features are raw March performance metrics. |

## 4. What I excluded and why

*The list of fields you refused to use \u2014 with one line of why each.*

In [ ]:
excluded = pd.DataFrame({
    "field": [
        "April 2026 performance columns",
        "fact_content_query_90d (query table)",
        "dim_content metadata fields",
        "trend_direction / trend_pct",
        "is_declining_label (from starter CSV)",
        "content_id / client_id (starter CSV)",
        "Product flags / action labels / priority scores",
        "Baseline scores from existing pipelines",
    ],
    "reason": [
        "Future data \u2014 occurs after the March decision moment",
        "Out of scope \u2014 query-mix features not in the five-feature contract",
        "Not in the five-feature contract \u2014 content metadata is excluded from this scope",
        "Label-derived \u2014 trend_direction produces the is_declining label, so trend_pct is the label source",
        "Not used \u2014 this notebook uses the warehouse, not the starter CSV",
        "Identifiers only \u2014 grouping/splits, never modeling features",
        "Decision-derived \u2014 encode existing-system decisions, circular if used as features",
        "Baseline scores are comparison targets, not model inputs",
    ],
})

print("Excluded fields:")
print(excluded.to_string(index=False))

## 5. Client-grouped vs random evaluation (out-of-sample)

The previous Precision@1000 results are **in-sample diagnostics** \u2014 scored on the same data used to define the features and proxy. They demonstrate the mechanical relationship but do NOT represent genuine predictive performance.

To test generalisation, I now evaluate on held-out data using two splits:
- **Random split**: test set is a random 20% of rows (may include rows from training clients)
- **Client-grouped split**: entire clients are held out (tests whether the rule generalises to unseen clients)

The honest score rule is `march_gsc_impressions / (march_gsc_clicks + 1)`. I evaluate Precision@1000 on the held-out portion.

In [ ]:
from sklearn.model_selection import train_test_split, GroupKFold
import numpy as np

# Base rate for reference
base_rate = feature_frame["opportunity_proxy"].mean()
print(f"Base rate (opportunity_proxy == 1): {base_rate:.1%}")
print()

# --- Random 80/20 split ---
train_rand, test_rand = train_test_split(
    feature_frame, test_size=0.2, random_state=42
)

print(f"Random split: train={len(train_rand):,}, test={len(test_rand):,}")

# Score the test set with the honest rule
test_rand = test_rand.copy()
test_rand["honest_score"] = test_rand["march_gsc_impressions"] / (test_rand["march_gsc_clicks"] + 1)

K = 1000
top_k_rand = test_rand.nlargest(K, "honest_score")
precision_rand = top_k_rand["opportunity_proxy"].mean()

print(f"\nRandom held-out Precision@{K}: {precision_rand:.3f}")
print(f"Base rate: {base_rate:.1%}")
print(f"Lift over base: {precision_rand / base_rate:.2f}x")

In [ ]:
# --- Client-grouped split (GroupKFold, 5 folds) ---
# Hold out entire clients to test cross-client generalisation
groups = feature_frame["client_hash_id"]
unique_clients = feature_frame["client_hash_id"].nunique()

print(f"Clients in dataset: {unique_clients}")
print(f"Using GroupKFold with min(n_splits, n_clients) groups")
print()

n_splits = min(5, unique_clients)
gkf = GroupKFold(n_splits=n_splits)

grouped_precisions = []
for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(feature_frame, groups=groups)):
    test_fold = feature_frame.iloc[test_idx].copy()
    train_fold = feature_frame.iloc[train_idx]
    
    held_out_clients = test_fold["client_hash_id"].nunique()
    train_clients = train_fold["client_hash_id"].nunique()
    
    test_fold["honest_score"] = (
        test_fold["march_gsc_impressions"] / (test_fold["march_gsc_clicks"] + 1)
    )
    
    actual_k = min(K, len(test_fold))
    top_k_fold = test_fold.nlargest(actual_k, "honest_score")
    prec = top_k_fold["opportunity_proxy"].mean()
    grouped_precisions.append(prec)
    
    print(f"Fold {fold_idx}: train_clients={train_clients}, held_out_clients={held_out_clients}, "
          f"test_rows={len(test_fold):,}, Precision@{actual_k}={prec:.3f}")

mean_grouped = np.mean(grouped_precisions)
print(f"\nClient-grouped mean Precision@{K}: {mean_grouped:.3f}")
print(f"Base rate: {base_rate:.1%}")
print(f"Lift over base: {mean_grouped / base_rate:.2f}x")

In [ ]:
print("=" * 60)
print("COMPARISON: RANDOM vs CLIENT-GROUPED SPLIT")
print("=" * 60)
print()
print(f"{'Split':<35} {'Precision@1000':>15} {'Base rate':>10} {'Lift':>8}")
print("-" * 70)
print(f"{'Random held-out':<35} {precision_rand:>15.3f} {base_rate:>9.1%} {precision_rand/base_rate:>7.2f}x")
print(f"{'Client-grouped (mean)':<35} {mean_grouped:>15.3f} {base_rate:>9.1%} {mean_grouped/base_rate:>7.2f}x")
print(f"{'Gap (random - grouped)':<35} {precision_rand - mean_grouped:>15.3f}")
print()
print("In-sample results from earlier sections are DIAGNOSTIC ONLY:")
print(f"  Leaked score (proxy as feature):    P@1000 = 1.000  |  Base rate: {base_rate:.1%}")
print(f"  Honest score (all 5 features):      P@1000 = 0.572  |  Base rate: {base_rate:.1%}")
print(f"  Honest score (without imp/clicks):  P@1000 = 0.090  |  Base rate: {base_rate:.1%}")
print()
print("They measure mechanical overlap, not predictive skill.")
print("The out-of-sample numbers above show actual generalisation performance.")
print("The gap between random and client-grouped splits quantifies")
print("how much performance depends on memorising client-specific patterns.")

In [ ]:
print("=" * 60)
print("OUT-OF-SAMPLE EVALUATION SUMMARY")
print("=" * 60)
print()
print("In-sample diagnostics (NOT predictive performance):")
print(f"  Leaked score (proxy as feature):    P@1000 = 1.000  |  Base rate: {base_rate:.1%}")
print(f"  Honest score (all 5 features):      P@1000 = 0.572  |  Base rate: {base_rate:.1%}")
print(f"  Honest score (without imp/clicks):  P@1000 = 0.090  |  Base rate: {base_rate:.1%}")
print()
print("Out-of-sample (genuinely held-out):")
print(f"  Random held-out:                    P@1000 = {precision_rand:.3f}  |  Base rate: {base_rate:.1%}")
print(f"  Client-grouped (mean):              P@1000 = {mean_grouped:.3f}  |  Base rate: {base_rate:.1%}")
print(f"  Gap (random - grouped):             {precision_rand - mean_grouped:.3f}")
print()
print("The in-sample 0.572 does NOT represent genuine predictive skill.")
print("It reflects the mechanical relationship between proxy inputs and the proxy itself.")
print("The out-of-sample numbers above show actual generalisation performance.")

## 6. ML-05 Conclusion

### What passed
- **No future leakage**: No April or May data enters the March feature vector. Source partition is March 2026 only.
- **No decision-derived features**: No product flags, action labels, priority scores, or existing-system outputs are used as features.
- **No query-table leakage**: The query table (`fact_content_query_90d`) is excluded from the feature contract.

### What leakage was found
- **Label-derived mechanical dependence**: `march_gsc_impressions` and `march_gsc_clicks` are direct inputs to the proxy definition `(impressions > 0) & (clicks == 0)`. Using them as features allows the model to rediscover the proxy rule. The leaked score (proxy as feature) achieves Precision@1000 = 1.000, confirming this.

### What did NOT leak
- `march_gsc_ctr`, `march_ga4_sessions`, and `march_scroll_events` are NOT components of the proxy. They can carry independent signal.
- No external future data, product flags, or decision artifacts enter the feature vector.

### What remains a methodological limitation
- **Overlapping window**: The proxy and all features are computed from the same March 1\u201331 window. This is not future leakage, but it creates a mechanical relationship that limits interpretation. The in-sample Precision@1000 of 0.572 (all features) does NOT represent genuine predictive skill \u2014 it partly reflects the proxy rediscovering its own inputs.
- **Incomplete client histories**: Some clients have `gsc_data_start` or `ga4_data_start` after 2026-03-01, meaning their March aggregates are truncated. This affects a portion of the feature frame.
- **GA4 coverage**: GA4 data is unavailable for ~65% of daily rows; zero-filled placeholders may mask real engagement differences.

### Which features are mechanically tied to the proxy
- `march_gsc_impressions` and `march_gsc_clicks`: **mechanically tied** \u2014 they are the proxy's direct inputs.
- `march_gsc_ctr`: derived from impressions and clicks, so **partially mechanically tied**.
- `march_ga4_sessions` and `march_scroll_events`: **not mechanically tied** \u2014 independent data source.

### Is the feature vector ready for W04?
The feature vector can be used in W04 **with the following restriction**: W04 must treat `march_gsc_impressions` and `march_gsc_clicks` as known proxy inputs, not as independent predictors. Any model trained on the full five-feature set must report **out-of-sample grouped metrics** (client-held-out) alongside in-sample numbers. The honest score without impressions/clicks (Precision@1000 = 0.090) provides a baseline for independent-signal-only performance. **Do not present the in-sample 0.572 as genuine predictive skill.**

## Self-check

Before you submit, confirm each line honestly:

- [\u2713] Every section above is filled \u2014 markdown thinking AND the code that backs it
- [\u2713] The notebook runs top to bottom with no errors (Runtime \u2192 Run all)
- [\u2713] No client names, URLs, or private queries anywhere
- [\u2713] My claims use careful words: observed, measured, directional, decision-support
- [\u2713] Committed to my repo under `work/notebooks/` \u2014 then submit your repo URL on the card. Done.